In [0]:
base = "/Volumes/workspace/default/retail"

sales = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/sales_clean.csv")
products = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/products (1).csv")

print("Sales rows:", sales.count())
print("Product rows:", products.count())
display(sales.limit(5))

Sales rows: 10000
Product rows: 5


sale_id,store_id,product_id,quantity,price,discount_pct,cost,sale_date,returned_qty
1,4,5,1,199.0,0,139.3,2025-02-02,0
2,1,2,5,59.5,5,193.38,2025-05-07,2
3,2,5,2,199.0,20,278.6,2025-09-20,0
4,1,4,5,74.25,0,222.75,2025-06-03,0
5,3,1,4,89.99,0,223.18,2025-03-03,0


In [0]:
from pyspark.sql import functions as F

sales_t = (sales
    .withColumn("sale_date", F.to_date("sale_date"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("discount_pct", F.col("discount_pct").cast("double"))
    .withColumn("cost", F.col("cost").cast("double"))
    .withColumn("returned_qty", F.col("returned_qty").cast("int"))
    .dropDuplicates(["sale_id"])
    .dropna(subset=["product_id", "quantity", "price"])
    .withColumn("revenue", F.round(F.col("quantity") * F.col("price") * (1 - F.col("discount_pct") / 100), 2))
    .withColumn("profit", F.round(F.col("revenue") - F.col("cost"), 2))
    .withColumn("profit_margin_pct",
                F.when(F.col("revenue") > 0, F.round(F.col("profit") / F.col("revenue") * 100, 2)).otherwise(0.0))
)

print("Rows:", sales_t.count())
display(sales_t.limit(5))

Rows: 10000


sale_id,store_id,product_id,quantity,price,discount_pct,cost,sale_date,returned_qty,revenue,profit,profit_margin_pct
12,5,5,1,199.0,0.0,139.3,2025-06-08,0,199.0,59.7,30.0
18,4,1,2,89.99,5.0,111.59,2025-05-12,0,170.98,59.39,34.74
38,4,2,2,59.5,0.0,77.35,2025-07-15,0,119.0,41.65,35.0
67,5,3,6,24.99,20.0,82.47,2025-12-04,0,119.95,37.48,31.25
70,5,2,6,59.5,20.0,232.05,2025-01-29,0,285.6,53.55,18.75


In [0]:
sales_enriched = sales_t.join(products, on="product_id", how="left")

# check: every sale should have found its product
print("Sales without a matching product:", sales_enriched.filter(F.col("product_name").isNull()).count())

# save as a Delta table
(sales_enriched.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.sales_enriched"))

print("Rows saved:", spark.table("workspace.default.sales_enriched").count())
display(sales_enriched.limit(5))

Sales without a matching product: 0
Rows saved: 10000


product_id,sale_id,store_id,quantity,price,discount_pct,cost,sale_date,returned_qty,revenue,profit,profit_margin_pct,product_name,category
5,12,5,1,199.0,0.0,139.3,2025-06-08,0,199.0,59.7,30.0,Smart Watch,Electronics
1,18,4,2,89.99,5.0,111.59,2025-05-12,0,170.98,59.39,34.74,Headphones,Electronics
2,38,4,2,59.5,0.0,77.35,2025-07-15,0,119.0,41.65,35.0,Coffee Maker,Home
3,67,5,6,24.99,20.0,82.47,2025-12-04,0,119.95,37.48,31.25,Yoga Mat,Sports
2,70,5,6,59.5,20.0,232.05,2025-01-29,0,285.6,53.55,18.75,Coffee Maker,Home


In [0]:
import os

category_metrics = (sales_enriched
    .groupBy("category")
    .agg(F.sum("quantity").alias("units_sold"),
         F.round(F.sum("revenue"), 2).alias("total_revenue"),
         F.round(F.sum("cost"), 2).alias("total_cost"),
         F.round(F.sum("profit"), 2).alias("total_profit"))
    .withColumn("profit_margin_pct", F.round(F.col("total_profit") / F.col("total_revenue") * 100, 2))
    .orderBy(F.col("profit_margin_pct").desc()))

# save as a Delta table
(category_metrics.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.category_profit_margin"))

# save as a CSV file for the dashboard
os.makedirs(f"{base}/output", exist_ok=True)
category_metrics.toPandas().to_csv(f"{base}/output/category_profit_margin.csv", index=False)

display(category_metrics)

category,units_sold,total_revenue,total_cost,total_profit,profit_margin_pct
Sports,3704,84985.52,50908.85,34076.67,40.1
Apparel,5304,360870.89,236293.2,124577.69,34.52
Home,7642,417563.15,295556.22,122006.93,29.22
Electronics,18562,2380571.43,1746449.44,634121.99,26.64


In [0]:
%sql
SELECT product_name,
       category,
       SUM(quantity)          AS units_sold,
       ROUND(SUM(revenue), 2) AS total_revenue
FROM workspace.default.sales_enriched
GROUP BY product_name, category
ORDER BY units_sold DESC
LIMIT 3

product_name,category,units_sold,total_revenue
Headphones,Electronics,10050,827983.38
Smart Watch,Electronics,8512,1552588.05
Coffee Maker,Home,7642,417563.15


In [0]:
top3 = spark.sql("""
    SELECT product_name, category,
           SUM(quantity) AS units_sold,
           ROUND(SUM(revenue), 2) AS total_revenue
    FROM workspace.default.sales_enriched
    GROUP BY product_name, category
    ORDER BY units_sold DESC
    LIMIT 3
""")

top3.toPandas().to_csv(f"{base}/output/top3_products.csv", index=False)
display(top3)

display(dbutils.fs.ls(f"{base}/output"))

product_name,category,units_sold,total_revenue
Headphones,Electronics,10050,827983.38
Smart Watch,Electronics,8512,1552588.05
Coffee Maker,Home,7642,417563.15


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/retail/output/category_profit_margin.csv,category_profit_margin.csv,270,1794909309000
dbfs:/Volumes/workspace/default/retail/output/top3_products.csv,top3_products.csv,159,1794909365000
